# HD-CNN — Results
Loads from `checkpoints/hdcnn_best.pt`. No training happens here.

In [ ]:
import sys
sys.path.insert(0, '../src')

import torch
import numpy as np
import matplotlib.pyplot as plt
from config import HDCNN_CKPT, BASELINE_CKPT, DEVICE
from models import HDCNN, build_resnet18
from train import load_model_weights
from plot import plot_history
from data import build_loaders
print(f'Device: {DEVICE}')

## Load HD-CNN checkpoint

In [ ]:
ckpt = torch.load(HDCNN_CKPT, map_location=DEVICE)
coarse_groups = ckpt['coarse_groups']
print(f"Best val acc:  {ckpt['best_val_acc']:.4f}")
print(f"Trained for:   {ckpt['epoch']+1} epochs")
print(f"Coarse groups: {len(coarse_groups)}")
for k, g in enumerate(coarse_groups):
    print(f"  Group {k:2d}: {len(g)} classes")

## Training curves

In [ ]:
plot_history(ckpt['history'],
             title='HD-CNN — Training and Validation Curves',
             save_name='hdcnn_curves.png')

## Compare HD-CNN vs Baseline

In [ ]:
baseline_ckpt = torch.load(BASELINE_CKPT, map_location='cpu')
hdcnn_ckpt    = torch.load(HDCNN_CKPT,    map_location='cpu')

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(baseline_ckpt['history']['val_accs'], label='Baseline ResNet18', linestyle='--')
ax.plot(hdcnn_ckpt['history']['val_accs'],    label='HD-CNN')
ax.set_xlabel('Epoch')
ax.set_ylabel('Validation Accuracy')
ax.set_title('HD-CNN vs Baseline — Validation Accuracy')
ax.legend()
plt.tight_layout()
plt.savefig('../figures/hdcnn_vs_baseline.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Baseline best val acc: {baseline_ckpt['best_val_acc']:.4f}")
print(f"HD-CNN   best val acc: {hdcnn_ckpt['best_val_acc']:.4f}")
delta = hdcnn_ckpt['best_val_acc'] - baseline_ckpt['best_val_acc']
print(f"Δ: {delta:+.4f}")

## Coarse routing analysis

In [ ]:
# Load model and run coarse routing on val set
hd_model = HDCNN(coarse_groups, num_fine=100, pretrained_backbone=False).to(DEVICE)
hd_model.load_state_dict(hdcnn_ckpt['best_state'])
hd_model.eval()

_, val_loader = build_loaders('standard')

# Collect coarse routing weights per sample
all_coarse = []
with torch.no_grad():
    for x, y in val_loader:
        x = x.to(DEVICE)
        _, coarse_probs, _ = hd_model(x)
        all_coarse.append(coarse_probs.cpu())

all_coarse = torch.cat(all_coarse, dim=0)  # (N, K)
mean_routing = all_coarse.mean(0).numpy()

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(range(len(mean_routing)), mean_routing)
ax.set_xlabel('Coarse group index')
ax.set_ylabel('Mean routing weight')
ax.set_title('Average coarse routing weights across val set')
ax.axhline(1/len(mean_routing), color='red', linestyle='--', label='Uniform')
ax.legend()
plt.tight_layout()
plt.savefig('../figures/hdcnn_routing.png', dpi=150, bbox_inches='tight')
plt.show()